# Vector Search - Qdrant Semantic Review Search
## Part 1: Setup & Connection

In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
import pandas as pd
import os

In [2]:
load_dotenv(find_dotenv(), override=True)

engine = create_engine(os.getenv("PG_CONNECTION"))

client = QdrantClient(host="localhost", port=6333)

model = SentenceTransformer("all-MiniLM-L6-v2")

print("PostgreSQL connected")
print("Qdrant persistent storage ready")
print("Embedding model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PostgreSQL connected
Qdrant persistent storage ready
Embedding model loaded


 ## Part 2: Load Reviews & Index (100k)

In [3]:
reviews = pd.read_sql("""
    SELECT 
        review_id, 
        product_id, 
        skin_type, 
        rating, 
        review_text
    FROM reviews
    WHERE review_text IS NOT NULL
    AND LENGTH(review_text) > 50
    LIMIT 100000;
""", engine)

print(f"Loaded {len(reviews):,} reviews")
print(f"Skin type distribution:")
print(reviews["skin_type"].value_counts())

Loaded 100,000 reviews
Skin type distribution:
skin_type
combination    47485
dry            15718
oily           11163
normal         10815
Name: count, dtype: int64


In [4]:
from qdrant_client.models import Distance, VectorParams

# Only create if it doesn't already exist
existing = [c.name for c in client.get_collections().collections]

if "sephora_reviews" not in existing:
    client.create_collection(
        collection_name="sephora_reviews",
        vectors_config=VectorParams(
            size=384,
            distance=Distance.COSINE
        )
    )
    print("Collection created")
else:
    print("Collection already exists - skipping creation")

Collection created


In [5]:
# Check how many are already indexed
collection_info = client.get_collection("sephora_reviews")
already_indexed = collection_info.points_count

print(f"Already indexed: {already_indexed:,}")
print(f"To index: {len(reviews) - already_indexed:,}")

if already_indexed < len(reviews):
    remaining = reviews.iloc[already_indexed:]
    texts = remaining["review_text"].tolist()

    print(f"\nEmbedding {len(texts):,} reviews...")
    print("Estimated time: 3-5 minutes")

    embeddings = model.encode(
        texts,
        batch_size=256,
        show_progress_bar=True
    )

    print("\nIndexing into Qdrant...")
    batch_size = 1000

    for i in range(0, len(remaining), batch_size):
        batch_df = remaining.iloc[i:i+batch_size]
        batch_emb = embeddings[i:i+batch_size]

        points = [
            PointStruct(
                id=int(batch_df.iloc[j]["review_id"]),
                vector=batch_emb[j].tolist(),
                payload={
                    "product_id": str(batch_df.iloc[j]["product_id"]),
                    "skin_type":  str(batch_df.iloc[j]["skin_type"]),
                    "rating":     float(batch_df.iloc[j]["rating"]),
                    "review_text": batch_df.iloc[j]["review_text"][:300]
                }
            )
            for j in range(len(batch_df))
        ]

        client.upsert(collection_name="sephora_reviews", points=points)

        if i % 10000 == 0:
            print(f"  Indexed {i + len(batch_df):,} / {len(remaining):,}")

    print(f"\nDone! Total indexed: {len(reviews):,} reviews")
else:
    print("All reviews already indexed - loading from disk")

Already indexed: 0
To index: 100,000

Embedding 100,000 reviews...
Estimated time: 3-5 minutes


Batches:   0%|          | 0/391 [00:00<?, ?it/s]


Indexing into Qdrant...
  Indexed 1,000 / 100,000
  Indexed 11,000 / 100,000
  Indexed 21,000 / 100,000
  Indexed 31,000 / 100,000
  Indexed 41,000 / 100,000
  Indexed 51,000 / 100,000
  Indexed 61,000 / 100,000
  Indexed 71,000 / 100,000
  Indexed 81,000 / 100,000
  Indexed 91,000 / 100,000

Done! Total indexed: 100,000 reviews


## Part 3: Basic Semantic Search

In [6]:
def semantic_search(query, top_k=5):
    query_vector = model.encode(query).tolist()

    results = client.query_points(
        collection_name="sephora_reviews",
        query=query_vector,
        limit=top_k
    ).points

    print(f"\nQuery: '{query}'")
    print("─" * 60)
    for r in results:
        print(f"Score    : {r.score:.3f}")
        print(f"Skin Type: {r.payload['skin_type']}")
        print(f"Rating   : {r.payload['rating']}")
        print(f"Review   : {r.payload['review_text'][:150]}")
        print("─" * 60)

semantic_search("great moisturizer for dry sensitive skin")


Query: 'great moisturizer for dry sensitive skin'
────────────────────────────────────────────────────────────
Score    : 0.865
Skin Type: dry
Rating   : 5.0
Review   : This is literally the best moisturizer for anyone with dry skin
────────────────────────────────────────────────────────────
Score    : 0.861
Skin Type: dry
Rating   : 5.0
Review   : I have very dry sensitive skin and this is the best moisturizer I have ever used. I have tried them all and this is sooooo GOOD! Love it!
────────────────────────────────────────────────────────────
Score    : 0.857
Skin Type: combination
Rating   : 5.0
Review   : As someone who struggles with sensitive/ dry skin this has been a life saving moisturizer! I’ve gone through so many brands but this one is amazing! B
────────────────────────────────────────────────────────────
Score    : 0.857
Skin Type: combination
Rating   : 5.0
Review   : As someone who struggles with sensitive/ dry skin this has been a life saving moisturizer! I’ve gone thr

## Part 4: Filtered Search by Skin Type

In [7]:
def filtered_search(query, skin_type, top_k=5):
    query_vector = model.encode(query).tolist()

    results = client.query_points(
        collection_name="sephora_reviews",
        query=query_vector,
        query_filter=Filter(
            must=[
                FieldCondition(
                    key="skin_type",
                    match=MatchValue(value=skin_type)
                )
            ]
        ),
        limit=top_k
    ).points

    print(f"\nQuery: '{query}' | Skin Type: {skin_type}")
    print("─" * 60)
    for r in results:
        print(f"Score  : {r.score:.3f}")
        print(f"Rating : {r.payload['rating']}")
        print(f"Review : {r.payload['review_text'][:150]}")
        print("─" * 60)

# Compare same query across skin types
query = "lightweight product that controls shine"
for skin in ["dry", "oily", "combination", "Normal"]:
    filtered_search(query, skin, top_k=2)


Query: 'lightweight product that controls shine' | Skin Type: dry
────────────────────────────────────────────────────────────
Score  : 0.490
Rating : 5.0
Review : I really love this lip balm. It’s hydrating but looks glossy, without being sticky! I love the glow it gives my lips and it compliments a simple makeu
────────────────────────────────────────────────────────────
Score  : 0.488
Rating : 5.0
Review : I absolutely love this product! I would recommend to anyone wanting a glow and lots of moisture!
────────────────────────────────────────────────────────────

Query: 'lightweight product that controls shine' | Skin Type: oily
────────────────────────────────────────────────────────────
Score  : 0.487
Rating : 4.0
Review : This is a very lightweight serum. Gives a dewy glow and absorbs quickly for immediate make up application.
────────────────────────────────────────────────────────────
Score  : 0.463
Rating : 5.0
Review : lightweight moisturizer that sinks right in and plumps up

## Part 5: Product Similarity Search

In [8]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

def find_similar_products(product_id, top_k=5):
    product_reviews = pd.read_sql(f"""
        SELECT review_id, review_text, rating
        FROM reviews
        WHERE product_id = '{product_id}'
        AND review_text IS NOT NULL
        LIMIT 10;
    """, engine)

    embeddings = model.encode(product_reviews["review_text"].tolist())
    product_vector = embeddings.mean(axis=0).tolist()

    # Exclude source product directly in Qdrant filter
    results = client.query_points(
        collection_name="sephora_reviews",
        query=product_vector,
        query_filter=Filter(
            must_not=[
                FieldCondition(
                    key="product_id",
                    match=MatchValue(value=product_id)
                )
            ]
        ),
        limit=50
    ).points

    # Get unique products
    seen = {}
    for r in results:
        pid = r.payload["product_id"]
        if pid not in seen:
            seen[pid] = r.score

    similar = list(seen.keys())[:top_k]

    if similar:
        placeholders = ",".join([f"'{p}'" for p in similar])
        product_names = pd.read_sql(f"""
            SELECT product_id, product_name, brand_name, price_usd
            FROM products
            WHERE product_id IN ({placeholders});
        """, engine)
        print(f"\nProducts similar to P420652 (Lip Sleeping Mask):")
        print("─" * 60)
        print(product_names.to_string())
    else:
        print("No similar products found in indexed subset")

find_similar_products("P420652")


Products similar to P420652 (Lip Sleeping Mask):
────────────────────────────────────────────────────────────
  product_id                     product_name           brand_name  price_usd
0     P12573  Intense Therapy Lip Balm SPF 25           Jack Black       10.0
1    P443563                   Lip Glowy Balm              LANEIGE       18.0
2     P42204                    Rosebud Salve  Rosebud Perfume Co.        7.0


In [9]:
# Step 1 - verify reviews exist
test = pd.read_sql("""
    SELECT review_id, review_text 
    FROM reviews 
    WHERE product_id = 'P420652'
    AND review_text IS NOT NULL
    LIMIT 5;
""", engine)

print(f"Reviews found: {len(test)}")
print(test.head())

Reviews found: 5
   review_id                                        review_text
0          2  I bought this lip mask after reading the revie...
1          3  My review title says it all! I get so excited ...
2          4  I’ve always loved this formula for a long time...
3          5  If you have dry cracked lips, this is a must h...
4          6  The scent isn’t my favourite but it works grea...


In [10]:
# Step 2 - check what product_ids look like inside Qdrant
results = client.query_points(
    collection_name="sephora_reviews",
    query=model.encode("lip mask").tolist(),
    limit=5
).points

for r in results:
    print(r.payload["product_id"])

P443563
P443563
P42204
P443563
P12573


## Part 6: Rating-Filtered Search

In [11]:
def search_by_rating(query, min_rating=4.0, top_k=5):
    from qdrant_client.models import Range

    query_vector = model.encode(query).tolist()

    results = client.query_points(
        collection_name="sephora_reviews",
        query=query_vector,
        query_filter=Filter(
            must=[
                FieldCondition(
                    key="rating",
                    range=Range(gte=min_rating)
                )
            ]
        ),
        limit=top_k
    ).points

    print(f"\nQuery: '{query}' | Min Rating: {min_rating}")
    print("─" * 60)
    for r in results:
        print(f"Score  : {r.score:.3f}")
        print(f"Rating : {r.payload['rating']}")
        print(f"Review : {r.payload['review_text'][:150]}")
        print("─" * 60)

search_by_rating("anti aging serum that actually works", min_rating=4.5)


Query: 'anti aging serum that actually works' | Min Rating: 4.5
────────────────────────────────────────────────────────────
Score  : 0.858
Rating : 5.0
Review : Bareminerals long life herb serum is a really good serum to use for anti aging than other products that I’ve tried.
────────────────────────────────────────────────────────────
Score  : 0.751
Rating : 5.0
Review : Glycolic Acid, hyaluronic acid, and niacinamide are three IMPORTANT ingredients for anti-aging. This serum is soo luxurious feeling and effective as i
────────────────────────────────────────────────────────────
Score  : 0.741
Rating : 5.0
Review : This is one of the best glycolic serums I’ve ever used! I have tried tons of anti-aging product and this is by far one of my favorites that I plan to 
────────────────────────────────────────────────────────────
Score  : 0.741
Rating : 5.0
Review : This anti aging serum is good and works well, I got this for free on pinchme
────────────────────────────────────────────────

## Collection Statistics

In [13]:
info = client.get_collection("sephora_reviews")

print("=" * 50)
print("QDRANT COLLECTION STATS")
print("=" * 50)
print(f"Total vectors indexed : {info.points_count:,}")
print(f"Vector dimensions     : 384")
print(f"Distance metric       : Cosine")
print(f"Storage               : Persistent (disk)")
print(f"Storage path          : localhost:6333")
print(f"Embedding model       : all-MiniLM-L6-v2")

QDRANT COLLECTION STATS
Total vectors indexed : 100,000
Vector dimensions     : 384
Distance metric       : Cosine
Storage               : Persistent (disk)
Storage path          : localhost:6333
Embedding model       : all-MiniLM-L6-v2


In [14]:
for skin in ["dry", "oily", "combination", "normal"]:
    query_vector = model.encode("hydrating product that works well").tolist()
    
    results = client.query_points(
        collection_name="sephora_reviews",
        query=query_vector,
        limit=50  # fetch more so we have enough to filter from
    ).points
    
    skin_results = [r for r in results 
                    if r.payload["skin_type"].lower() == skin.lower()]
    
    print(f"\nSkin type: {skin} — {len(skin_results)} matches")
    for r in skin_results:
        print(f"  Score: {r.score:.3f} | {r.payload['review_text'][:100]}")


Skin type: dry — 16 matches
  Score: 0.816 | Quick way to hydrate.  A bit pricey but a little goes a long way and it is very effective.
  Score: 0.816 | Quick way to hydrate.  A bit pricey but a little goes a long way and it is very effective.
  Score: 0.737 | Super hydrating. Helps with dry patches.  A little goes a long way. Worth the money.
  Score: 0.737 | Super hydrating. Helps with dry patches.  A little goes a long way. Worth the money.
  Score: 0.695 | Ultra hydrating. Smells amazing. Would highly recommend for dry skin.
  Score: 0.695 | Ultra hydrating. Smells amazing. Would highly recommend for dry skin.
  Score: 0.679 | Works well hydrated skin very well  I highly recommend this product
  Score: 0.679 | Works well hydrated skin very well  I highly recommend this product
  Score: 0.670 | Love this product! So light yet so hydrating at the same time. Works like magic. Also, works perfect
  Score: 0.670 | Love this product! So light yet so hydrating at the same time. Works lik